# Chapter 10 — Introduction to RL: Live Example — Grid World

**Session 4 | Chapter 10 | Duration: ~10 min**

We build a simple grid world from scratch to understand the RL concepts:
states, actions, rewards, and the difference between a random vs optimal agent.

**Key principle:** An RL agent learns by interacting with an environment — no labeled data, only trial-and-error with reward signals.

> **Note:** This example requires only numpy and matplotlib — no external RL library needed.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.colors import ListedColormap
import seaborn as sns

sns.set_theme(style='white')
np.random.seed(42)
print('Ready!')

## 1. The Grid World Environment

A 4×4 grid with:
- **S** = Start (top-left)
- **G** = Goal (bottom-right) → reward +10
- **H** = Holes → reward -5 (game over)
- **F** = Free cells → reward -0.1 (small step penalty to encourage efficiency)

In [ ]:
# Grid layout
# 0 = free, 1 = hole, 2 = goal, 3 = start
GRID = np.array([
    [3, 0, 0, 0],
    [0, 1, 0, 1],
    [0, 0, 0, 1],
    [1, 0, 0, 2]
])

ROWS, COLS = GRID.shape
N_STATES   = ROWS * COLS            # 16 states
N_ACTIONS  = 4                      # 0=Left, 1=Down, 2=Right, 3=Up
ACTION_MAP = {0: 'Left', 1: 'Down', 2: 'Right', 3: 'Up'}
MOVE_DR    = {0: (0,-1), 1: (1,0), 2: (0,1), 3: (-1,0)}  # (row, col) deltas

REWARDS    = {0: -0.1,   # free cell (small step penalty)
              1: -5.0,   # hole
              2: +10.0,  # goal
              3: -0.1}   # start

def visualize_grid(agent_pos=None, title='Grid World'):
    fig, ax = plt.subplots(figsize=(5, 5))
    labels = {0: 'F', 1: 'H', 2: 'G', 3: 'S'}
    colors = {0: '#ecf0f1', 1: '#e74c3c', 2: '#2ecc71', 3: '#3498db'}
    
    for r in range(ROWS):
        for c in range(COLS):
            cell_type = GRID[r, c]
            color = colors[cell_type]
            rect = mpatches.FancyBboxPatch([c, ROWS-1-r], 1, 1,
                                            boxstyle='round,pad=0.05',
                                            facecolor=color, edgecolor='white', linewidth=2)
            ax.add_patch(rect)
            ax.text(c + 0.5, ROWS-1-r + 0.5, labels[cell_type],
                    ha='center', va='center', fontsize=18, fontweight='bold', color='#2c3e50')
    
    if agent_pos is not None:
        r, c = agent_pos
        ax.text(c + 0.5, ROWS-1-r + 0.5, '🤖', ha='center', va='center', fontsize=22)
    
    ax.set_xlim(0, COLS)
    ax.set_ylim(0, ROWS)
    ax.set_aspect('equal')
    ax.axis('off')
    ax.set_title(title, fontsize=13)
    plt.tight_layout()
    plt.show()

visualize_grid(agent_pos=(0, 0), title='Grid World Environment\n(S=Start, F=Free, H=Hole, G=Goal)')

## 2. Environment Functions

These implement the RL environment interface.

In [ ]:
def pos_to_state(r, c):
    return r * COLS + c

def state_to_pos(s):
    return s // COLS, s % COLS

def env_reset():
    return pos_to_state(0, 0)  # start at top-left

def env_step(state, action):
    r, c = state_to_pos(state)
    dr, dc = MOVE_DR[action]
    nr, nc = r + dr, c + dc
    
    # Stay in bounds
    if not (0 <= nr < ROWS and 0 <= nc < COLS):
        nr, nc = r, c  # hit wall, stay
    
    next_state = pos_to_state(nr, nc)
    cell_type  = GRID[nr, nc]
    reward     = REWARDS[cell_type]
    done       = cell_type in [1, 2]  # hole or goal ends episode
    
    return next_state, reward, done

print('Environment functions defined!')
print(f'Number of states:  {N_STATES}')
print(f'Number of actions: {N_ACTIONS}')
print(f'Actions: {ACTION_MAP}')

## 3. Random Agent — Bumbling Around

An agent that takes completely random actions. This is our baseline (and it's bad).

In [ ]:
def run_episode(policy_fn, max_steps=100):
    state = env_reset()
    trajectory = [state_to_pos(state)]
    total_reward = 0
    
    for step in range(max_steps):
        action = policy_fn(state)
        next_state, reward, done = env_step(state, action)
        total_reward += reward
        trajectory.append(state_to_pos(next_state))
        state = next_state
        if done:
            break
    
    return trajectory, total_reward

# Random policy
random_policy = lambda s: np.random.randint(N_ACTIONS)

# Run many random episodes
n_eval_episodes = 1000
successes = 0
random_rewards = []

for _ in range(n_eval_episodes):
    traj, total_r = run_episode(random_policy)
    random_rewards.append(total_r)
    # Check if reached goal (bottom-right = (3,3))
    if traj[-1] == (3, 3):
        successes += 1

print(f'Random Agent over {n_eval_episodes} episodes:')
print(f'  Success rate:    {successes/n_eval_episodes:.1%}')
print(f'  Average reward:  {np.mean(random_rewards):.2f}')

# Show one example trajectory
traj_ex, _ = run_episode(random_policy, max_steps=20)
print(f'\nExample trajectory: {traj_ex[:10]}...' if len(traj_ex) > 10 else f'\nTrajectory: {traj_ex}')

## 4. Optimal Agent — Hand-Crafted Policy

Let's see how much better a correct policy is.  
In Ch11, Q-Learning will learn this automatically!

In [ ]:
# Optimal policy: hand-crafted to navigate to goal
# Actions: 0=Left, 1=Down, 2=Right, 3=Up
OPTIMAL_POLICY = {
    0: 2,  # (0,0) Start → Right
    1: 2,  # (0,1) → Right
    2: 1,  # (0,2) → Down
    3: 1,  # (0,3) → Down
    4: 2,  # (1,0) → Right (can't go to hole at (1,1))
    5: 1,  # (1,1) Hole (never reached)
    6: 1,  # (1,2) → Down
    7: 1,  # (1,3) Hole
    8: 2,  # (2,0) → Right
    9: 2,  # (2,1) → Right
   10: 1,  # (2,2) → Down
   11: 1,  # (2,3) Hole
   12: 2,  # (3,0) Hole
   13: 2,  # (3,1) → Right
   14: 2,  # (3,2) → Right
   15: 2,  # (3,3) Goal
}

optimal_policy_fn = lambda s: OPTIMAL_POLICY[s]

optimal_rewards = []
for _ in range(n_eval_episodes):
    _, total_r = run_episode(optimal_policy_fn)
    optimal_rewards.append(total_r)

print(f'Optimal Policy over {n_eval_episodes} episodes:')
print(f'  Success rate:    100%')
print(f'  Average reward:  {np.mean(optimal_rewards):.2f}')

print(f'\nRandom Agent average reward:  {np.mean(random_rewards):.2f}')
print(f'Optimal Policy average reward: {np.mean(optimal_rewards):.2f}')
print(f'→ The learned policy is {np.mean(optimal_rewards)/max(np.mean(random_rewards), 0.01):.1f}× better!')

In [ ]:
# Visualize the optimal path
opt_traj, opt_reward = run_episode(optimal_policy_fn)

fig, ax = plt.subplots(figsize=(5, 5))
labels = {0: 'F', 1: 'H', 2: 'G', 3: 'S'}
colors_cell = {0: '#ecf0f1', 1: '#e74c3c', 2: '#2ecc71', 3: '#3498db'}

for r in range(ROWS):
    for c in range(COLS):
        ct = GRID[r, c]
        rect = mpatches.FancyBboxPatch([c, ROWS-1-r], 1, 1, boxstyle='round,pad=0.05',
                                        facecolor=colors_cell[ct], edgecolor='white', linewidth=2)
        ax.add_patch(rect)
        ax.text(c+0.5, ROWS-1-r+0.5, labels[ct], ha='center', va='center',
                fontsize=14, fontweight='bold', color='#2c3e50')

# Draw trajectory
for i, (r, c) in enumerate(opt_traj):
    ax.text(c+0.5, ROWS-1-r+0.5, str(i), ha='center', va='center',
            fontsize=9, color='navy', fontweight='bold')

# Draw arrows
for i in range(len(opt_traj)-1):
    r1, c1 = opt_traj[i]
    r2, c2 = opt_traj[i+1]
    ax.annotate('', xy=(c2+0.5, ROWS-1-r2+0.5), xytext=(c1+0.5, ROWS-1-r1+0.5),
                arrowprops=dict(arrowstyle='->', color='navy', lw=2))

ax.set_xlim(0, COLS)
ax.set_ylim(0, ROWS)
ax.set_aspect('equal')
ax.axis('off')
ax.set_title(f'Optimal Path (reward={opt_reward:.1f})\nNumbers show step order', fontsize=12)
plt.tight_layout()
plt.show()

## Summary

| Concept | What we saw |
|---------|------------|
| State | Grid cell position |
| Action | Move left/down/right/up |
| Reward | +10 goal, -5 hole, -0.1 step |
| Episode | One run from start to goal/hole |
| Random policy | ~low success rate, poor reward |
| Optimal policy | 100% success, maximum reward |

**The RL question:** Can the agent learn the optimal policy *automatically* from experience?  
**Chapter 11 answer:** Yes — with Q-Learning!
